# ⚡ CircuitAI — Colab GPU Backend
**Step 1**: Runtime → Change runtime type → T4 or L4 GPU

**Step 2**: Run cells in order.

**Step 3**: Copy the `trycloudflare.com` URL into your local CircuitAI frontend.

In [ ]:
# ── Cell 0: Mount Drive & Set Working Directory ─────────────────
import sys, os
from google.colab import drive

drive.mount('/content/drive')

# ⚠️ Update this path if your folder is in a different location
BACKEND_DIR = '/content/drive/MyDrive/knowledge_graph/backend'

os.chdir(BACKEND_DIR)
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Files found: {os.listdir()}')

In [ ]:
# ── Cell 1: Install system dependencies ────────────────────────
# Note: poppler is NO LONGER needed (replaced by pypdfium2 which is pure Python)
# Only Cloudflared needs a system install
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb
print('Cloudflared installed ✓')

In [ ]:
# ── Cell 2: Install Python dependencies ────────────────────────
!pip install -r requirements.txt -q
print('Python deps installed ✓')

In [ ]:
# ── Cell 3: Pre-load model (keeps subsequent requests fast) ────
from model_loader import load_model
load_model()
print('Model ready ✓')

In [ ]:
# ── Cell 4: Start FastAPI server + Cloudflare Tunnel ───────────
import subprocess, threading, time, uvicorn

# Import the app OBJECT (not a string) — avoids module-not-found on reload
from main import app

def start_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(3)  # Give uvicorn time to bind

# Start Cloudflare tunnel and capture the public URL
print('Starting Cloudflare tunnel...')
p = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in p.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        tokens = line.split()
        public_url = next((t for t in tokens if 'trycloudflare.com' in t), None)
        if public_url:
            print('\n' + '='*60)
            print(f'  🔥 PUBLIC URL: {public_url}')
            print('  Paste this into CircuitAI frontend > Connection field')
            print('='*60)
        break